# Module 3 - Class 5: Pipelines

A reusable churn pipeline with preprocessing, modeling, and joblib export.


In [ ]:
# === SETUP — run this first ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
print('Loaded:', df.shape)


### Cell 1 - set up the target and features


In [ ]:
# Cell 1 - set up the target and features.
df['Churn_bin'] = df['Churn'].map({'No': 0, 'Yes': 1})
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_features = ['gender', 'Contract', 'PaymentMethod']
X = df[num_features + cat_features]
y = df['Churn_bin']
print('X:', X.shape, 'y:', y.shape)


### Cell 2 - split into train and test


In [ ]:
# Cell 2 - split into train and test.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')


### Cell 3 - build the numeric preprocessor


In [ ]:
# Cell 3 - build the numeric preprocessor.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
num_pipe


### Cell 4 - build the categorical preprocessor


In [ ]:
# Cell 4 - build the categorical preprocessor.
from sklearn.preprocessing import OneHotEncoder
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
cat_pipe


### Cell 5 - combine both with ColumnTransformer


In [ ]:
# Cell 5 - combine both with ColumnTransformer.
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
])
preprocessor


### Cell 6 - full pipeline: preprocessor + model


In [ ]:
# Cell 6 - full pipeline: preprocessor + model.
from sklearn.linear_model import LogisticRegression
full_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])
full_pipe


### Cell 7 - fit on training data


In [ ]:
# Cell 7 - fit on training data.
full_pipe.fit(X_train, y_train)
print('✅ Pipeline trained')


### Cell 8 - evaluate on test data


In [ ]:
# Cell 8 - evaluate on test data.
from sklearn.metrics import accuracy_score
y_pred = full_pipe.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc:.4f}')


### Cell 9 - save the trained pipeline


In [ ]:
# Cell 9 - save the trained pipeline.
import joblib
joblib.dump(full_pipe, 'churn_pipeline.joblib')
loaded = joblib.load('churn_pipeline.joblib')
print('Saved and loaded.')
print('Test accuracy from loaded model:', round(loaded.score(X_test, y_test), 4))
